In [1]:
%use dataframe
// %use lets-plot
// %use kandy
// %use plotly

USE {
	dependencies("org.duckdb:duckdb_jdbc:$1.5.5.1")
}

import kotlinx.serialization.json.Json

In [2]:
val dir = "/Users/darronporter/Downloads/Dissertation/code/output/results/gcloud_results-2026-08-17-run_5"

val mcts_easy_vs_mcts_easy_records = "/Users/darronporter/Downloads/Dissertation/code/output/results/gcloud_results-2026-08-17-run_8/agent-1_MCTS_EASY_agent-2_MCTS_EASY_games_record.json"

// MCTS Easy as Agent 1
val mcts_easy_vs_mcts_random_records = "$dir/agent-1_MCTS_EASY_agent-2_MCTS_RANDOM_games_record.json"
val mcts_easy_vs_minimax_easy_records = "$dir/agent-1_MCTS_EASY_agent-2_MINIMAX_EASY_games_record.json"
val mcts_easy_vs_minimax_greedy_records = "$dir/agent-1_MCTS_EASY_agent-2_MINIMAX_GREEDY_games_record.json"

// MCTS Random as Agent 1
val mcts_random_vs_mcts_easy_records = "$dir/agent-1_MCTS_RANDOM_agent-2_MCTS_EASY_games_record.json"
val mcts_random_vs_minimax_easy_records = "$dir/agent-1_MCTS_RANDOM_agent-2_MINIMAX_EASY_games_record.json"
val mcts_random_vs_minimax_greedy_records = "$dir/agent-1_MCTS_RANDOM_agent-2_MINIMAX_GREEDY_games_record.json"

// Minimax Easy as Agent 1
val minimax_easy_vs_mcts_easy_records = "$dir/agent-1_MINIMAX_EASY_agent-2_MCTS_EASY_games_record.json"
val minimax_easy_vs_mcts_random_records = "$dir/agent-1_MINIMAX_EASY_agent-2_MCTS_RANDOM_games_record.json"
val minimax_easy_vs_minimax_greedy_records = "$dir/agent-1_MINIMAX_EASY_agent-2_MINIMAX_GREEDY_games_record.json"

// Minimax Greedy as Agent 1
val minimax_greedy_vs_mcts_easy_records = "$dir/agent-1_MINIMAX_GREEDY_agent-2_MCTS_EASY_games_record.json"
val minimax_greedy_vs_mcts_random_records = "$dir/agent-1_MINIMAX_GREEDY_agent-2_MCTS_RANDOM_games_record.json"
val minimax_greedy_vs_minimax_easy_records = "$dir/agent-1_MINIMAX_GREEDY_agent-2_MINIMAX_EASY_games_record.json"

In [4]:
val mcts_e_vs_mcts_r = DataFrame.readJson(mcts_easy_vs_mcts_random_records)
val mcts_e_vs_mm_e = DataFrame.readJson(mcts_easy_vs_minimax_easy_records)
val mcts_e_vs_mm_g = DataFrame.readJson(mcts_easy_vs_minimax_greedy_records)
val mcts_r_vs_mcts_e = DataFrame.readJson(mcts_random_vs_mcts_easy_records)
val mcts_r_vs_mm_e = DataFrame.readJson(mcts_random_vs_minimax_easy_records)
val mcts_r_vs_mm_g = DataFrame.readJson(mcts_random_vs_minimax_greedy_records)
val mm_e_vs_mcts_e = DataFrame.readJson(minimax_easy_vs_mcts_easy_records)
val mm_e_vs_mcts_r = DataFrame.readJson(minimax_easy_vs_mcts_random_records)
val mm_e_vs_mm_g = DataFrame.readJson(minimax_easy_vs_minimax_greedy_records)
val mm_g_vs_mcts_e = DataFrame.readJson(minimax_greedy_vs_mcts_easy_records)
val mm_g_vs_mcts_r = DataFrame.readJson(minimax_greedy_vs_mcts_random_records)
val mm_g_vs_mm_e = DataFrame.readJson(minimax_greedy_vs_minimax_easy_records)

In [5]:
val mcts_e_vs_mcts_e = DataFrame.readJson(mcts_easy_vs_mcts_easy_records).cast<SearchInfo>()

In [5]:
val dataframes =
    listOf<DataFrame<*>>(
        mcts_e_vs_mcts_r,
        mcts_e_vs_mm_e,
        mcts_e_vs_mm_g,
        mcts_r_vs_mcts_e,
        mcts_r_vs_mm_e,
        mcts_r_vs_mm_g,
        mm_e_vs_mcts_e,
        mm_e_vs_mcts_r,
        mm_e_vs_mm_g,
        mm_g_vs_mcts_e,
        mm_g_vs_mcts_r,
        mm_g_vs_mm_e,
    )

In [ ]:
mcts_e_vs_mcts_r.generateInterfaces("Matrx")

In [6]:
@DataSchema
interface Matrx {
	val gameId: Int
	val gameState: GameState
	val seed: Long
	val timestamp: Long
	val turn: Int
	val winner: Winner

	@DataSchema(isOpen = false)
	interface Bitboard {
		val blackDVONNLayer: List<Long>
		val blackGIPF: Long?
		val blackPUNCTLayer: List<Long>
		val blackPotentials: Long
		val blackTAMSK: Long?
		val blackYINSH: Long?
		val blackZERTZ: Long?
		val whiteDVONNLayer: List<Long>
		val whiteGIPF: Long?
		val whitePUNCTLayer: List<Long>
		val whitePotentials: Long
		val whiteTAMSK: Long?
		val whiteYINSH: Long?
		val whiteZERTZ: Long?
	}

	@DataSchema(isOpen = false)
	interface CurrentPlayer {
		val capturedPieces: List<Int>
		val model: String
		val name: String
		val piecesInReserve: List<Int>
		val strength: String
		val timeControl: Boolean
		val useRAVE: Boolean
	}

	@DataSchema(isOpen = false)
	interface GameState {
		val bitboard: Bitboard
		val currentPlayer: CurrentPlayer
		val nextPlayer: CurrentPlayer
	}

	@DataSchema(isOpen = false)
	interface Winner {
		val loser: CurrentPlayer
		val winCondition: String
		val winner: CurrentPlayer
	}
}

In [ ]:
dataframes.map { it.head(5)  }

In [ ]:
mcts_e_vs_mcts_r

In [7]:
var concat = dataframes.concat().cast<Matrx>()

In [ ]:
concat.select {
	winner.winCondition and winner.winner.cols(0..2)
}.valueCounts()

In [ ]:
dataframes.map { it.count() }

In [ ]:
dataframes.map { it.count() }.sum()

In [8]:
val totalGames = 3000.0

In [9]:
val wins = concat.select {
	winner.winCondition and winner.winner.select { model and strength }
}.valueCounts()

In [10]:
wins.add("win_percentage") {
	(count / totalGames) * 100.0
}

winCondition,model,strength,count,win_percentage
MadeTheLastMove,MCTS,EASY,2123,70.766667
MadeTheLastMove,MINIMAX,EASY,1569,52.300000
MadeTheLastMove,MINIMAX,GREEDY,1414,47.133333
CapturedAllGIPFPieces,MCTS,EASY,567,18.900000
CapturedAllGIPFPieces,MINIMAX,EASY,194,6.466667
MadeTheLastMove,MCTS,RANDOM,66,2.200000
CapturedAllGIPFPieces,MINIMAX,GREEDY,40,1.333333
CapturedAllGIPFPieces,MCTS,RANDOM,27,0.900000


In [11]:
val winning = concat.select {
	winner.winner.select { model and strength }
}.valueCounts()

In [12]:
winning.add("win_percentage") {
	(count / totalGames) * 100.0
}

model,strength,count,win_percentage
MCTS,EASY,2690,89.666667
MINIMAX,EASY,1763,58.766667
MINIMAX,GREEDY,1454,48.466667
MCTS,RANDOM,93,3.100000


In [13]:
// 1. Isolate the winner columns (including the unique seed)
val winners = concat.select {
	seed and winner.winCondition
}.rename { colsOf<String>() }.into { "winner_${it.name}" }

// 2. Isolate and rename the current player columns
val currentPlayers = concat.select {
	seed and winner.winner.select { name and model and strength }
}.rename { colsOf<String>() }.into { "winner_${it.name}" }

// 3. Isolate and rename the next player columns
val nextPlayers = concat.select {
	seed and winner.loser.select { name and model and strength }
}.rename { colsOf<String>() }.into { "loser_${it.name}" }

// 4. Join them all together on the 'seed' column
winners
//	.join(currentPlayers, "seed")
	.join(nextPlayers, "seed")
	.remove("seed") // Optional: remove if you don't want seed included in the valueCounts
	.valueCounts()

winner_winCondition,loser_name,loser_model,loser_strength,count
MadeTheLastMove,WHITE,MCTS,RANDOM,1385
MadeTheLastMove,BLACK,MCTS,RANDOM,1370
MadeTheLastMove,WHITE,MINIMAX,GREEDY,658
MadeTheLastMove,BLACK,MINIMAX,GREEDY,531
MadeTheLastMove,WHITE,MINIMAX,EASY,529
MadeTheLastMove,BLACK,MINIMAX,EASY,408
CapturedAllGIPFPieces,BLACK,MINIMAX,GREEDY,194
MadeTheLastMove,WHITE,MCTS,EASY,180
CapturedAllGIPFPieces,WHITE,MINIMAX,EASY,171
CapturedAllGIPFPieces,WHITE,MINIMAX,GREEDY,163


In [15]:
// 1. Isolate the winner columns (including the unique seed)
val winners = concat.select {
	seed and winner.winner.select { name and model and strength }
}.rename { colsOf<String?>() }.into { "winner_${it.name}" }

// 3. Isolate and rename the next player columns
val nextPlayers = concat.select {
	seed and winner.loser.select { name and model and strength }
}.rename { colsOf<String?>() }.into { "loser_${it.name}" }

// 4. Join them all together on the 'seed' column
winners
//	.join(currentPlayers, "seed")
	.join(nextPlayers, "seed")
	.remove("seed") // Optional: remove if you don't want seed included in the valueCounts
	.valueCounts()

winner_name,winner_model,winner_strength,loser_name,loser_model,loser_strength,count
WHITE,MCTS,EASY,BLACK,MCTS,RANDOM,499
BLACK,MCTS,EASY,WHITE,MCTS,RANDOM,499
BLACK,MINIMAX,EASY,WHITE,MCTS,RANDOM,481
WHITE,MINIMAX,EASY,BLACK,MCTS,RANDOM,480
BLACK,MINIMAX,GREEDY,WHITE,MCTS,RANDOM,474
WHITE,MINIMAX,GREEDY,BLACK,MCTS,RANDOM,474
BLACK,MCTS,EASY,WHITE,MINIMAX,GREEDY,447
WHITE,MCTS,EASY,BLACK,MINIMAX,GREEDY,446
BLACK,MCTS,EASY,WHITE,MINIMAX,EASY,433
WHITE,MCTS,EASY,BLACK,MINIMAX,EASY,366


In [16]:
// 1. Isolate the winner columns (including the unique seed)
val winners = concat.select {
	seed and winner.winner.cols(1..2)
}.rename { colsOf<String?>() }.into { "winner_${it.name}" }

// 2. Isolate and rename the current player columns
val currentPlayers = concat.select {
	seed and gameState.currentPlayer.select { model and strength }
}.rename { colsOf<String>() }.into { "current_${it.name}" }

// 3. Isolate and rename the next player columns
val nextPlayers = concat.select {
	seed and winner.loser.select { model and strength }
}.rename { colsOf<String>() }.into { "loser_${it.name}" }

// 4. Join them all together on the 'seed' column
winners
//	.join(currentPlayers, "seed")
	.join(nextPlayers, "seed")
	.remove("seed") // Optional: remove if you don't want seed included in the valueCounts
	.valueCounts()

winner_model,winner_strength,loser_model,loser_strength,count
MCTS,EASY,MCTS,RANDOM,998
MINIMAX,EASY,MCTS,RANDOM,961
MINIMAX,GREEDY,MCTS,RANDOM,948
MCTS,EASY,MINIMAX,GREEDY,893
MCTS,EASY,MINIMAX,EASY,799
MINIMAX,EASY,MINIMAX,GREEDY,601
MINIMAX,GREEDY,MINIMAX,EASY,399
MINIMAX,EASY,MCTS,EASY,201
MINIMAX,GREEDY,MCTS,EASY,107
MCTS,RANDOM,MINIMAX,GREEDY,52


In [17]:
// 1. Isolate the winner columns (including the unique seed)
val winners = concat.select {
	seed and turn and winner.winCondition and winner.winner.select { model and strength }
}.rename { colsOf<String?>() }.into { "winner_${it.name}" }

// 3. Isolate and rename the next player columns
val nextPlayers = concat.select {
	seed and winner.loser.select { model and strength }
}.rename { colsOf<String?>() }.into { "loser_${it.name}" }

// 4. Join them all together on the 'seed' column
winners
//	.join(currentPlayers, "seed")
	.join(nextPlayers, "seed")
	.remove("seed") // Optional: remove if you don't want seed included in the valueCounts
	.valueCounts()

winner_winCondition,winner_model,winner_strength,loser_model,loser_strength,count
MadeTheLastMove,MINIMAX,GREEDY,MCTS,RANDOM,939
MadeTheLastMove,MCTS,EASY,MCTS,RANDOM,924
MadeTheLastMove,MINIMAX,EASY,MCTS,RANDOM,892
MadeTheLastMove,MCTS,EASY,MINIMAX,GREEDY,660
MadeTheLastMove,MCTS,EASY,MINIMAX,EASY,539
MadeTheLastMove,MINIMAX,EASY,MINIMAX,GREEDY,495
MadeTheLastMove,MINIMAX,GREEDY,MINIMAX,EASY,368
CapturedAllGIPFPieces,MCTS,EASY,MINIMAX,EASY,260
CapturedAllGIPFPieces,MCTS,EASY,MINIMAX,GREEDY,233
MadeTheLastMove,MINIMAX,EASY,MCTS,EASY,182


In [ ]:
mm_e_vs_mcts_e.turn.values().plus(mcts_e_vs_mm_e.turn.values()).average()
mm_e_vs_mcts_e.turn.values().plus(mcts_e_vs_mm_e.turn.values()).max()

In [ ]:
mm_e_vs_mcts_e.turn.values().plus(mcts_e_vs_mm_e.turn.values()).max()

In [18]:
concat.turn.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
turn,Int,6000,138,0,55,158,68.961667,19.957962,20,55.000000,66.000000,80.000000,240


In [35]:
concat.select { turn and winner.winner.model and winner.winner.strength and winner.loser.model.rename and winner.loser.strength } .groupBy { "model" and "strength" }.mean()

java.lang.IllegalStateException: Column not found: loserModel

In [23]:
concat.turn.mean() + 1.96 * concat.turn.std().div(sqrt(6000.0))

69.46667278456603

In [42]:
// 1. Isolate the winner columns (including the unique seed)
val winners = concat.select {
	seed and turn and winner.winner.select { model and strength }
}.rename { colsOf<String?>() }.into { "winner_${it.name}" }

// 3. Isolate and rename the next player columns
val nextPlayers = concat.select {
	seed and winner.loser.select { model and strength }
}.rename { colsOf<String?>() }.into { "loser_${it.name}" }

// 4. Join them all together on the 'seed' column
winners
//	.join(currentPlayers, "seed")
	.join(nextPlayers, "seed")
	.remove("seed") // Optional: remove if you don't want seed included in the valueCounts
	.groupBy { "winner_model" and "winner_strength" and "loser_model" and "loser_strength" }.mean()

winner_model,winner_strength,loser_model,loser_strength,turn
MCTS,EASY,MCTS,RANDOM,55.397796
MCTS,RANDOM,MCTS,EASY,47.500000
MINIMAX,EASY,MCTS,EASY,79.507463
MCTS,EASY,MINIMAX,EASY,79.117647
MCTS,EASY,MINIMAX,GREEDY,72.640538
MINIMAX,GREEDY,MCTS,EASY,75.719626
MINIMAX,EASY,MCTS,RANDOM,67.431842
MCTS,RANDOM,MINIMAX,EASY,83.256410
MCTS,RANDOM,MINIMAX,GREEDY,78.538462
MINIMAX,GREEDY,MCTS,RANDOM,69.638186


In [17]:
// Assuming `gamesDf` is your DataFrame<SearchInfo>
val turnDurationsLong = mcts_e_vs_mcts_e
	.select {
		seed and gameId and
				gameState.turnDuration
	}
	// Step 1: Unpack the nested gameState.turnDuration object
	.flatten()

turnDurationsLong

<iframe onload="o_resize_iframe_out_8()" style="width:100%;" class="result_container" id="iframe_out_8" frameBorder="0" srcdoc=" <html>
 <head>
 <style type="text/css">
 :root {
 --background: #fff;
 --background-odd: #f5f5f5;
 --background-hover: #d9edfd;
 --header-text-color: #474747;
 --text-color: #848484;
 --text-color-dark: #000;
 --text-color-medium: #737373;
 --text-color-pale: #b3b3b3;
 --inner-border-color: #aaa;
 --bold-border-color: #000;
 --link-color: #296eaa;
 --link-color-pale: #296eaa;
 --link-hover: #1a466c;
}

:root[theme="dark"], :root [data-jp-theme-light="false"], .dataframe_dark{
 --background: #303030;
 --background-odd: #3c3c3c;
 --background-hover: #464646;
 --header-text-color: #dddddd;
 --text-color: #b3b3b3;
 --text-color-dark: #dddddd;
 --text-color-medium: #b2b2b2;
 --text-color-pale: #737373;
 --inner-border-color: #707070;
 --bold-border-color: #777777;
 --link-color: #008dc0;
 --link-color-pale: #97e1fb;
 --link-hover: #00688e;
}

p.dataframe_description {
 color: var(--text-color-dark);
}

table.dataframe {
 font-family: "Helvetica Neue", Helvetica, Arial, sans-serif;
 font-size: 12px;
 background-color: var(--background);
 color: var(--text-color-dark);
 border: none;
 border-collapse: collapse;
}

table.dataframe th, td {
 padding: 6px;
 border: 1px solid transparent;
 text-align: left;
}

table.dataframe th {
 background-color: var(--background);
 color: var(--header-text-color);
}

table.dataframe td {
 vertical-align: top;
 white-space: nowrap;
}

table.dataframe th.bottomBorder {
 border-bottom-color: var(--bold-border-color);
}

table.dataframe tbody > tr:nth-child(odd) {
 background: var(--background-odd);
}

table.dataframe tbody > tr:nth-child(even) {
 background: var(--background);
}

table.dataframe tbody > tr:hover {
 background: var(--background-hover);
}

table.dataframe a {
 cursor: pointer;
 color: var(--link-color);
 text-decoration: none;
}

table.dataframe tr:hover > td a {
 color: var(--link-color-pale);
}

table.dataframe a:hover {
 color: var(--link-hover);
 text-decoration: underline;
}

table.dataframe img {
 max-width: fit-content;
}

table.dataframe th.complex {
 background-color: var(--background);
 border: 1px solid var(--background);
}

table.dataframe .leftBorder {
 border-left-color: var(--inner-border-color);
}

table.dataframe .rightBorder {
 border-right-color: var(--inner-border-color);
}

table.dataframe .rightAlign {
 text-align: right;
}

table.dataframe .expanderSvg {
 width: 8px;
 height: 8px;
 margin-right: 3px;
}

table.dataframe .expander {
 display: flex;
 align-items: center;
}

/* formatting */

table.dataframe .null {
 color: var(--text-color-pale);
}

table.dataframe .structural {
 color: var(--text-color-medium);
 font-weight: bold;
}

table.dataframe .dataFrameCaption {
 font-weight: bold;
}

table.dataframe .numbers {
 color: var(--text-color-dark);
}

table.dataframe td:hover .formatted .structural, .null {
 color: var(--text-color-dark);
}

table.dataframe tr:hover .formatted .structural, .null {
 color: var(--text-color-dark);
}


:root {
 --scroll-bg: #f5f5f5;
 --scroll-fg: #b3b3b3;
}
:root[theme="dark"], :root [data-jp-theme-light="false"]{
 --scroll-bg: #3c3c3c;
 --scroll-fg: #97e1fb;
}
body {
 scrollbar-color: var(--scroll-fg) var(--scroll-bg);
}
body::-webkit-scrollbar {
 width: 10px; /* Mostly for vertical scrollbars */
 height: 10px; /* Mostly for horizontal scrollbars */
}
body::-webkit-scrollbar-thumb {
 background-color: var(--scroll-fg);
}
body::-webkit-scrollbar-track {
 background-color: var(--scroll-bg);
}
 </style>
 </head>
 <body>
 <table class="dataframe" id="df_-822083570"></table>

<p class="dataframe_description">... showing only top 20 of 103 rows</p><p class="dataframe_description">DataFrame: rowsCount = 103, columnsCount = 143</p>

 </body>
 <script>
 (function () {
 window.DataFrame = window.DataFrame || new (function () {
 this.addTable = function (df) {
 let cols = df.cols;
 for (let i = 0; i < cols.length; i++) 

In [14]:
// Extract and unpivot turn search info
val turnSearchInfoLong = mcts_e_vs_mcts_e
	.select {
//		gameId and
				gameState.turnSearchInfo
	}
	.flatten()
//	.unfold("1")
//	.convert("turnNumber").toInt()
//	// Step 1: Explode List<DataFrame<*>> so each row holds a single DataFrame<*>
//	.explode("searchFramesList")
//	.rename("searchFramesList").into("searchData")
//	// Step 2: Unpack or concatenate the search data frame into tabular columns
//	.flatten()
//	.sortBy { gameId and "turnNumber" }

turnSearchInfoLong

<iframe onload="o_resize_iframe_out_5()" style="width:100%;" class="result_container" id="iframe_out_5" frameBorder="0" srcdoc=" <html>
 <head>
 <style type="text/css">
 :root {
 --background: #fff;
 --background-odd: #f5f5f5;
 --background-hover: #d9edfd;
 --header-text-color: #474747;
 --text-color: #848484;
 --text-color-dark: #000;
 --text-color-medium: #737373;
 --text-color-pale: #b3b3b3;
 --inner-border-color: #aaa;
 --bold-border-color: #000;
 --link-color: #296eaa;
 --link-color-pale: #296eaa;
 --link-hover: #1a466c;
}

:root[theme="dark"], :root [data-jp-theme-light="false"], .dataframe_dark{
 --background: #303030;
 --background-odd: #3c3c3c;
 --background-hover: #464646;
 --header-text-color: #dddddd;
 --text-color: #b3b3b3;
 --text-color-dark: #dddddd;
 --text-color-medium: #b2b2b2;
 --text-color-pale: #737373;
 --inner-border-color: #707070;
 --bold-border-color: #777777;
 --link-color: #008dc0;
 --link-color-pale: #97e1fb;
 --link-hover: #00688e;
}

p.dataframe_description {
 color: var(--text-color-dark);
}

table.dataframe {
 font-family: "Helvetica Neue", Helvetica, Arial, sans-serif;
 font-size: 12px;
 background-color: var(--background);
 color: var(--text-color-dark);
 border: none;
 border-collapse: collapse;
}

table.dataframe th, td {
 padding: 6px;
 border: 1px solid transparent;
 text-align: left;
}

table.dataframe th {
 background-color: var(--background);
 color: var(--header-text-color);
}

table.dataframe td {
 vertical-align: top;
 white-space: nowrap;
}

table.dataframe th.bottomBorder {
 border-bottom-color: var(--bold-border-color);
}

table.dataframe tbody > tr:nth-child(odd) {
 background: var(--background-odd);
}

table.dataframe tbody > tr:nth-child(even) {
 background: var(--background);
}

table.dataframe tbody > tr:hover {
 background: var(--background-hover);
}

table.dataframe a {
 cursor: pointer;
 color: var(--link-color);
 text-decoration: none;
}

table.dataframe tr:hover > td a {
 color: var(--link-color-pale);
}

table.dataframe a:hover {
 color: var(--link-hover);
 text-decoration: underline;
}

table.dataframe img {
 max-width: fit-content;
}

table.dataframe th.complex {
 background-color: var(--background);
 border: 1px solid var(--background);
}

table.dataframe .leftBorder {
 border-left-color: var(--inner-border-color);
}

table.dataframe .rightBorder {
 border-right-color: var(--inner-border-color);
}

table.dataframe .rightAlign {
 text-align: right;
}

table.dataframe .expanderSvg {
 width: 8px;
 height: 8px;
 margin-right: 3px;
}

table.dataframe .expander {
 display: flex;
 align-items: center;
}

/* formatting */

table.dataframe .null {
 color: var(--text-color-pale);
}

table.dataframe .structural {
 color: var(--text-color-medium);
 font-weight: bold;
}

table.dataframe .dataFrameCaption {
 font-weight: bold;
}

table.dataframe .numbers {
 color: var(--text-color-dark);
}

table.dataframe td:hover .formatted .structural, .null {
 color: var(--text-color-dark);
}

table.dataframe tr:hover .formatted .structural, .null {
 color: var(--text-color-dark);
}


:root {
 --scroll-bg: #f5f5f5;
 --scroll-fg: #b3b3b3;
}
:root[theme="dark"], :root [data-jp-theme-light="false"]{
 --scroll-bg: #3c3c3c;
 --scroll-fg: #97e1fb;
}
body {
 scrollbar-color: var(--scroll-fg) var(--scroll-bg);
}
body::-webkit-scrollbar {
 width: 10px; /* Mostly for vertical scrollbars */
 height: 10px; /* Mostly for horizontal scrollbars */
}
body::-webkit-scrollbar-thumb {
 background-color: var(--scroll-fg);
}
body::-webkit-scrollbar-track {
 background-color: var(--scroll-bg);
}
 </style>
 </head>
 <body>
 <table class="dataframe" id="df_-822083576"></table>

<p class="dataframe_description">... showing only top 20 of 103 rows</p><p class="dataframe_description">DataFrame: rowsCount = 103, columnsCount = 141</p>

 </body>
 <script>
 (function () {
 window.DataFrame = window.DataFrame || new (function () {
 this.addTable = function (df) {
 let cols = df.cols;
 for (let i = 0; i < cols.length; i++) 